# Construction des caractéristiques par session

Objectif : représenter chaque session par des informations permettant de distinguer les utilisateurs.

| Informations à construire | Représentation |
| --- | --- |
| Caractéristiques générales | Comptages, diversité et proportions |
| Types d’actions | Nombre d’occurrences par session |
| Écrans, configurations et chaînes | Nombre de mentions par session |
| Navigateur | Catégorie conservée pour l’encodage ultérieur |

La cible sera `y = util`. Les entrées `X` ne contiendront ni `util` ni `session_id`. Les encodages appris seront ajustés après la séparation entre apprentissage et validation.

In [1]:
from pathlib import Path
from collections import Counter
import re

import pandas as pd
from IPython.display import display

DATA_PATH = (
    Path(r"C:\Users\Rog strix Scar\Desktop\BE_MachineLearning")
    / "data"
    / "train.csv"
)

print("Fichier trouvé :", DATA_PATH.is_file())

Fichier trouvé : True


## Chargement des sessions

Les données brutes sont conservées dans `df_raw`. Une copie de travail, `df_train`, est créée avec des listes de traces indépendantes.

In [2]:
sessions = []

with DATA_PATH.open(encoding="utf-8-sig") as fichier:
    for numero_ligne, ligne in enumerate(fichier, start=1):
        champs = ligne.rstrip("\r\n").split(",")

        if len(champs) < 3:
            raise ValueError(
                f"Ligne {numero_ligne} : format inattendu"
            )

        sessions.append({
            "session_id": numero_ligne,
            "util": champs[0],
            "navigateur": champs[1],
            "traces": champs[2:],
        })

df_raw = pd.DataFrame(sessions)

df_train = df_raw.copy(deep=True)
df_train["traces"] = df_raw["traces"].apply(list)

print("Nombre de sessions :", len(df_train))
print("Nombre d’utilisateurs :", df_train["util"].nunique())

display(df_train.head())

Nombre de sessions : 3279
Nombre d’utilisateurs : 247


,session_id,util,navigateur,traces
0,1,nuh,Firefox,[Création d'un écran(infologic.core.accueil.Ac...
1,2,muz,Google Chrome,[Création d'un écran(infologic.core.gui.contro...
2,3,zrx,Microsoft Edge,[Affichage d'une dialogue(infologic.core.gui.c...
3,4,pou,Firefox,[Création d'un écran(infologic.core.gui.contro...
4,5,ald,Google Chrome,[Affichage d'une dialogue(infologic.acti.modul...


## Fonctions d’extraction

| Élément | Traitement |
| --- | --- |
| `tXX` | Reconnaître un marqueur temporel |
| Nom d’action | Séparer le nom de son contexte et du suffixe de modification |
| `(…)`, `<…>`, `$…$` | Extraire les contextes, configurations et chaînes |

In [3]:
pattern_ecran = re.compile(r"\((.*?)\)")
pattern_configuration = re.compile(r"<(.*?)>")
pattern_chaine = re.compile(r"\$(.*?)\$")


def est_marqueur_temporel(trace):
    return re.fullmatch(r"t\d+", trace) is not None


def extraire_type_action(action):
    nom = re.split(r"[(<$]", action, maxsplit=1)[0].strip()

    if nom.endswith("1"):
        nom = nom[:-1].rstrip()

    return nom


def extraire_contextes(actions, pattern):
    valeurs = []

    for action in actions:
        valeurs.extend(pattern.findall(action))

    return valeurs

## Extraction des informations par session

Les listes d’actions, de temps, de types d’actions et de contextes sont construites pour chaque session. Les traces originales restent conservées.

In [4]:
df_train["actions"] = df_train["traces"].apply(
    lambda traces: [
        trace for trace in traces
        if not est_marqueur_temporel(trace)
    ]
)

df_train["temps"] = df_train["traces"].apply(
    lambda traces: [
        int(trace[1:])
        for trace in traces
        if est_marqueur_temporel(trace)
    ]
)

df_train["types_actions"] = df_train["actions"].apply(
    lambda actions: [
        extraire_type_action(action)
        for action in actions
    ]
)

df_train["ecrans_contextes"] = df_train["actions"].apply(
    lambda actions: extraire_contextes(actions, pattern_ecran)
)

df_train["configurations"] = df_train["actions"].apply(
    lambda actions: extraire_contextes(actions, pattern_configuration)
)

df_train["chaines"] = df_train["actions"].apply(
    lambda actions: extraire_contextes(actions, pattern_chaine)
)

display(
    df_train[
        ["util", "navigateur", "types_actions", "ecrans_contextes",
         "configurations", "chaines"]
    ].head()
)

,util,navigateur,types_actions,ecrans_contextes,configurations,chaines
0,nuh,Firefox,"[Création d'un écran, Affichage d'une dialogue...","[infologic.core.accueil.AccueilController, inf...","[DEF_03/24, ACCUEIL_GMAO, 001, ACCUEIL_GMAO, D...","[JCP, JCP, JCP, JCP, JCP, JCP, JCP, JCP, JCP, ..."
1,muz,Google Chrome,"[Création d'un écran, Création d'un écran, Sél...",[infologic.core.gui.controllers.BlankControlle...,"[TELEM_MINIMAL, DEF_03/24, TELEM_MINIMAL, DEF_...","[TEC, TEC, DB, DB, DB, DB, DB, DB, DB, DB, DB,..."
2,zrx,Microsoft Edge,"[Affichage d'une dialogue, Exécution d'un bout...",[infologic.core.gui.controllers.BlankControlle...,"[ACCUEIL_INST, INSTALLATEUR, DEF_07/23, ACCUEI...","[GP, GP, GP, GP, GP, MES, MES, MES, MES, MES, ..."
3,pou,Firefox,"[Création d'un écran, Exécution d'un bouton, A...",[infologic.core.gui.controllers.nested.InputFo...,"[STD, DEF_03/24, STD, DEF_03/24, STD, DEF_03/2...","[JCP, JCP, JCP, JCP, JCP, JCP, JCP, JCP, JCP, ..."
4,ald,Google Chrome,"[Affichage d'une dialogue, Exécution d'un bout...",[infologic.acti.modules.AT_ACTIVITES.compteren...,"[INSTALLATEUR, DEF_03/24, TELEM_MINIMAL, DEF_0...","[ST, ST, ST, GP, GP, GP, GP, GP, GP, GP, GP]"


## Caractéristiques générales

| Caractéristique | Signification |
| --- | --- |
| `nb_actions` | Nombre d’actions |
| `nb_marqueurs_temporels` | Nombre de marqueurs, répétitions incluses |
| `dernier_repere_s` | Plus grand repère temporel enregistré |
| `sans_actions` | 1 si aucune action, sinon 0 |
| `sans_marqueur_temporel` | 1 si aucun marqueur, sinon 0 |
| `nb_actions_modification` | Nombre d’actions terminées par `1` |
| `part_actions_modification` | Proportion des actions en modification |
| `nb_types_actions_distincts` | Diversité des types d’actions |
| Diversité des contextes | Nombre de valeurs distinctes par catégorie |

Pour une session sans action, les proportions sont fixées à zéro
et l’indicateur `sans_actions` conserve cette information.

Si aucun marqueur temporel n’existe, le dernier repère est fixé à zéro
et l’indicateur `sans_marqueur_temporel` signale son absence.

In [5]:
df_train["nb_actions"] = df_train["actions"].apply(len)
df_train["nb_marqueurs_temporels"] = df_train["temps"].apply(len)

df_train["dernier_repere_s"] = df_train["temps"].apply(
    lambda temps: max(temps) if temps else 0
)

df_train["sans_actions"] = (
    df_train["nb_actions"].eq(0).astype(int)
)

df_train["sans_marqueur_temporel"] = (
    df_train["nb_marqueurs_temporels"].eq(0).astype(int)
)

df_train["nb_actions_modification"] = df_train["actions"].apply(
    lambda actions: sum(
        action.endswith("1") for action in actions
    )
)

df_train["part_actions_modification"] = (
    df_train["nb_actions_modification"]
    / df_train["nb_actions"].replace(0, float("nan"))
).fillna(0)

df_train["nb_types_actions_distincts"] = (
    df_train["types_actions"].apply(lambda valeurs: len(set(valeurs)))
)

df_train["nb_ecrans_contextes_distincts"] = (
    df_train["ecrans_contextes"].apply(lambda valeurs: len(set(valeurs)))
)

df_train["nb_configurations_distinctes"] = (
    df_train["configurations"].apply(lambda valeurs: len(set(valeurs)))
)

df_train["nb_chaines_distinctes"] = (
    df_train["chaines"].apply(lambda valeurs: len(set(valeurs)))
)

colonnes_numeriques = [
    "nb_actions",
    "nb_marqueurs_temporels",
    "dernier_repere_s",
    "sans_actions",
    "sans_marqueur_temporel",
    "nb_actions_modification",
    "part_actions_modification",
    "nb_types_actions_distincts",
    "nb_ecrans_contextes_distincts",
    "nb_configurations_distinctes",
    "nb_chaines_distinctes",
]

display(df_train[["util"] + colonnes_numeriques].head())

,util,nb_actions,nb_marqueurs_temporels,dernier_repere_s,sans_actions,sans_marqueur_temporel,nb_actions_modification,part_actions_modification,nb_types_actions_distincts,nb_ecrans_contextes_distincts,nb_configurations_distinctes,nb_chaines_distinctes
0,nuh,2514,614,2905,0,0,4,0.001591,19,7,3,2
1,muz,90,46,230,0,0,26,0.288889,11,5,2,2
2,zrx,608,158,750,0,0,50,0.082237,21,17,7,2
3,pou,886,313,1445,0,0,64,0.072235,19,10,4,1
4,ald,173,60,275,0,0,2,0.011561,14,8,4,2


## Comptages par session

Chaque liste est transformée en un dictionnaire de comptages. L’affichage présente les types d’actions de la première session, du plus fréquent au moins fréquent.

Le navigateur reste conservé dans sa colonne d’origine ; il n’est pas une action à compter.

In [6]:
df_train["comptages_actions"] = (
    df_train["types_actions"].apply(Counter)
)

df_train["comptages_ecrans"] = (
    df_train["ecrans_contextes"].apply(Counter)
)

df_train["comptages_configurations"] = (
    df_train["configurations"].apply(Counter)
)

df_train["comptages_chaines"] = (
    df_train["chaines"].apply(Counter)
)

session = df_train.iloc[0]

print("Session :", session["session_id"])
print("Utilisateur :", session["util"])

display(
    pd.DataFrame(
        session["comptages_actions"].most_common(),
        columns=["type_action", "nombre"],
    )
)

Session : 1
Utilisateur : nuh


,type_action,nombre
0,Exécution d'un bouton,698
1,Fermeture d'une dialogue,369
2,Affichage d'une dialogue,350
3,Affichage d'un toast,311
4,Saisie dans un champ,198
5,Filtrage / Tri,161
6,Lancement d'une action générique,124
7,Double-clic,87
8,Lancement d'une stat,85
9,Création d'un écran,72


In [7]:
def calculer_proportions(comptages):
    total = sum(comptages.values())

    if total == 0:
        return {}

    return {
        nom: nombre / total
        for nom, nombre in comptages.items()
    }


df_train["proportions_actions"] = (
    df_train["comptages_actions"].apply(calculer_proportions)
)

session = df_train.iloc[0]

resume_actions = pd.DataFrame({
    "nombre": dict(session["comptages_actions"]),
    "proportion": session["proportions_actions"],
})

display(
    resume_actions.sort_values("nombre", ascending=False).head(10)
)

,nombre,proportion
Exécution d'un bouton,698,0.277645
Fermeture d'une dialogue,369,0.146778
Affichage d'une dialogue,350,0.139220
Affichage d'un toast,311,0.123707
Saisie dans un champ,198,0.078759
Filtrage / Tri,161,0.064041
Lancement d'une action générique,124,0.049324
Double-clic,87,0.034606
Lancement d'une stat,85,0.033811
Création d'un écran,72,0.028640


In [8]:
def calculer_proportions(comptages):
    total = sum(comptages.values())

    if total == 0:
        return {}

    return {
        nom: nombre / total
        for nom, nombre in comptages.items()
    }


df_train["proportions_actions"] = (
    df_train["comptages_actions"].apply(calculer_proportions)
)

session = df_train.iloc[0]

resume_actions = pd.DataFrame({
    "nombre": dict(session["comptages_actions"]),
    "proportion": session["proportions_actions"],
})

display(
    resume_actions.sort_values("nombre", ascending=False).head(10)
)

,nombre,proportion
Exécution d'un bouton,698,0.277645
Fermeture d'une dialogue,369,0.146778
Affichage d'une dialogue,350,0.139220
Affichage d'un toast,311,0.123707
Saisie dans un champ,198,0.078759
Filtrage / Tri,161,0.064041
Lancement d'une action générique,124,0.049324
Double-clic,87,0.034606
Lancement d'une stat,85,0.033811
Création d'un écran,72,0.028640


In [9]:
def construire_caracteristiques(session):
    caracteristiques = {
        nom: session[nom]
        for nom in colonnes_numeriques
    }

    caracteristiques["navigateur"] = session["navigateur"]

    groupes = [
        ("action_nombre", session["comptages_actions"]),
        ("action_proportion", session["proportions_actions"]),
        ("ecran_nombre", session["comptages_ecrans"]),
        ("configuration_nombre", session["comptages_configurations"]),
        ("chaine_nombre", session["comptages_chaines"]),
    ]

    for prefixe, valeurs in groupes:
        for nom, valeur in valeurs.items():
            caracteristiques[f"{prefixe}::{nom}"] = valeur

    return caracteristiques


X = df_train.apply(construire_caracteristiques, axis=1)
X.name = "caracteristiques"

y = df_train["util"].copy()

print("Nombre de sessions dans X :", len(X))
print("Nombre de réponses dans y :", len(y))
print("Nombre de classes :", y.nunique())

display(
    pd.DataFrame({
        "util": y.head(),
        "nb_caracteristiques_presentes": X.head().apply(len),
    })
)

Nombre de sessions dans X : 3279
Nombre de réponses dans y : 3279
Nombre de classes : 247


,util,nb_caracteristiques_presentes
0,nuh,62
1,muz,43
2,zrx,80
3,pou,65
4,ald,54


In [10]:
caracteristiques_session = pd.Series(
    X.iloc[0],
    name="valeur",
)

print("Utilisateur réel :", y.iloc[0])

display(caracteristiques_session.to_frame().head(25))

Utilisateur réel : nuh


,valeur
nb_actions,2514
nb_marqueurs_temporels,614
dernier_repere_s,2905
sans_actions,0
sans_marqueur_temporel,0
nb_actions_modification,4
part_actions_modification,0.001591
nb_types_actions_distincts,19
nb_ecrans_contextes_distincts,7
nb_configurations_distinctes,3


In [11]:
assert len(X) == len(y) == len(df_raw)
assert X.index.equals(y.index)

assert all(
    "util" not in caracteristiques
    and "session_id" not in caracteristiques
    for caracteristiques in X
)

assert (
    df_train["comptages_actions"].apply(
        lambda comptages: sum(comptages.values())
    ) == df_train["nb_actions"]
).all()

sommes_proportions = df_train["proportions_actions"].apply(
    lambda proportions: sum(proportions.values())
)

avec_actions = df_train["nb_actions"].gt(0)

assert (
    (sommes_proportions[avec_actions] - 1).abs() < 1e-9
).all()

assert sommes_proportions[~avec_actions].eq(0).all()

assert df_train["part_actions_modification"].between(0, 1).all()

assert df_raw.equals(df_train[df_raw.columns])

assert all(
    originales is not copie
    for originales, copie in zip(
        df_raw["traces"], df_train["traces"]
    )
)

print("Tous les contrôles sont validés.")

Tous les contrôles sont validés.


## Bilan de la construction des caractéristiques

| Élément | État |
| --- | --- |
| Unité d’analyse | Une session |
| Caractéristiques générales | Calculées |
| Types d’actions | Comptages et proportions par session |
| Écrans, configurations et chaînes | Comptages par session |
| Navigateur | Conservé comme catégorie |
| Entrées `X` | Un dictionnaire de caractéristiques par session |
| Cible `y` | Identifiant de l’utilisateur |
| `util` et `session_id` dans `X` | Exclus |
| Données brutes | Conservées |
| Modèle | Pas encore entraîné |
